# LeetCode 983 — Minimum Cost For Tickets bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-for-tickets/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        ...
```

Ghi chú: bạn biết trước các ngày đi tàu trong năm (`days`, tăng ngặt, từ `1` đến `365`) và có ba loại vé: **1 ngày**, **7 ngày**, **30 ngày** (vé `k` ngày mua vào ngày `d` đi được các ngày `d, d+1, ..., d+k-1`). Tìm **tổng chi phí nhỏ nhất** để đi đủ mọi ngày. Đây là bài **quy hoạch động** kinh điển: trạng thái là "đã lo xong tới ngày nào", và quyết định là "chọn loại vé nào để phủ ngày hiện tại". Bẫy lớn của Python là **chỉ số âm**: `dp[d - 7]` khi `d < 7` sẽ lấy phần tử ở **cuối** mảng thay vì báo lỗi.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: phủ tập ngày bằng các đoạn liên tiếp độ dài `1`, `7`, `30` với chi phí khác nhau; chi phí **không** nhất thiết tăng theo độ dài vé.
2. Nắm các công cụ Python cần thiết: `set` để tra ngày đi tàu, `functools.lru_cache` cho đệ quy có nhớ, `bisect` để nhảy qua các ngày đã phủ, `max(0, ...)` để chặn chỉ số âm, và `heapq` (Dijkstra) cho oracle.
3. Hiểu nguyên lý quy hoạch động: `dp[d]` = chi phí nhỏ nhất để đi đủ mọi ngày đi tàu **không vượt quá ngày `d`**; nếu `d` không phải ngày đi thì `dp[d] = dp[d-1]`, nếu phải đi thì thử ba loại vé kết thúc ở ngày `d`.
4. Đi qua 4 cách: oracle Dijkstra trên tập ngày đã phủ (đúng theo định nghĩa, cho `n` nhỏ), đệ quy có nhớ theo chỉ số ngày đi tàu, DP theo lịch từng ngày (cách nộp chính), và DP chỉ trên các ngày đi tàu với tìm kiếm nhị phân.
5. Tránh các lỗi kinh điển: `dp[d - 7]` với `d < 7` (chỉ số âm), quên `dp[d] = dp[d - 1]` ở ngày không đi, **tham lam** chọn vé rẻ nhất theo giá mỗi ngày, lệch một đơn vị ở độ dài vé, và bỏ sót vé 30 ngày.

## 2. Đề bài & ví dụ

Bạn lên kế hoạch đi tàu trước một năm. Các ngày đi được cho bởi mảng `days`, mỗi ngày là số nguyên từ `1` đến `365`.

Vé tàu bán theo **ba cách**:

- vé **1 ngày** giá `costs[0]` đô la,
- vé **7 ngày** giá `costs[1]` đô la,
- vé **30 ngày** giá `costs[2]` đô la.

Vé cho phép đi **liên tiếp** đúng số ngày đó. Ví dụ mua vé 7 ngày vào ngày `2` thì đi được các ngày `2, 3, 4, 5, 6, 7, 8`.

Trả về **số đô la ít nhất** để đi đủ mọi ngày trong `days`.

**Ví dụ 1:** `days = [1,4,6,7,8,20]`, `costs = [2,7,15]` -> `11`. Ngày `1` mua vé 1 ngày (2), ngày `3` mua vé 7 ngày (7) phủ `3..9` (có `4, 6, 7, 8`), ngày `20` mua vé 1 ngày (2). Tổng `11`.

**Ví dụ 2:** `days = [1,2,3,4,5,6,7,8,9,10,30,31]`, `costs = [2,7,15]` -> `17`. Ngày `1` mua vé 30 ngày (15) phủ `1..30`, ngày `31` mua vé 1 ngày (2). Tổng `17`.

**Ràng buộc:**

- `1 <= days.length <= 365`
- `1 <= days[i] <= 365`
- `days` tăng ngặt.
- `costs.length == 3`
- `1 <= costs[i] <= 1000`

In [1]:
from typing import List, Dict, Tuple
from functools import lru_cache
from bisect import bisect_left, bisect_right
import heapq
import random, time

examples = [
    ([1, 4, 6, 7, 8, 20], [2, 7, 15], 11),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [2, 7, 15], 17),
]
for days, costs, expected in examples:
    print(f"days={days} costs={costs} -> kỳ vọng {expected}")

days=[1, 4, 6, 7, 8, 20] costs=[2, 7, 15] -> kỳ vọng 11
days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31] costs=[2, 7, 15] -> kỳ vọng 17


## 3. Kiến thức nền

### 3.1. Quy hoạch động theo lịch

Ta xét từng ngày `d = 1, 2, ..., 365`. Gọi `dp[d]` là chi phí nhỏ nhất để đi đủ **mọi ngày đi tàu `<= d`**. Có hai trường hợp:

- `d` **không** phải ngày đi tàu: không cần vé nào mới: `dp[d] = dp[d - 1]`.
- `d` **là** ngày đi tàu: ngày này phải được phủ bởi một vé; xét vé **kết thúc** (hoặc bao phủ tới) ngày `d`: vé `k` ngày phủ các ngày `d-k+1 .. d`, trước đó cần `dp[d - k]`. Vậy `dp[d] = min(dp[d-1] + c1, dp[d-7] + c7, dp[d-30] + c30)` với chỉ số `< 0` được hiểu là `0`.

Vì mua vé ở ngày bắt đầu cũng giống mua vé "đến hết ngày `d`" khi ta chỉ quan tâm các ngày đi, nên tính theo cách này không mất tổng quát.

In [2]:
costs = [2, 7, 15]
days = [1, 4, 6, 7, 8, 20]
travel = set(days)
dp = [0] * 21
for d in range(1, 21):
    if d not in travel:
        dp[d] = dp[d - 1]
    else:
        dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1], dp[max(0, d - 30)] + costs[2])
print("dp[1..20] =", dp[1:])
print("đáp án:", dp[20])
assert dp[20] == 11

dp[1..20] = [2, 2, 2, 4, 4, 6, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 11]
đáp án: 11


### 3.2. Chặn chỉ số âm bằng `max(0, d - k)`

Khi `d < 7`, `d - 7 < 0` và trong Python `dp[d - 7]` là `dp[-1]`, `dp[-2]`, ...: các phần tử **cuối** của mảng (với `dp` mới khởi tạo `dp[-1] = 0` nên có thể **tình cờ đúng**, nhưng không đáng tin). Dùng `max(0, d - 7)` để luôn đúng nghĩa "chưa có ngày nào trước đó, chi phí `0`".

In [3]:
dp_demo = [0, 5, 9, 4, 7, 6, 8, 3]
print("dp_demo[3 - 7] =", dp_demo[3 - 7], "(thực chất là dp_demo[-4])", "| dp_demo[max(0, 3 - 7)] =", dp_demo[max(0, 3 - 7)])

dp_demo[3 - 7] = 7 (thực chất là dp_demo[-4]) | dp_demo[max(0, 3 - 7)] = 0


### 3.3. Đệ quy có nhớ (`lru_cache`)

`@lru_cache(None)` lưu kết quả theo tham số; các lời gọi lặp lại tra bảng trong `O(1)`. Tham số phải **băm được** (số nguyên là được).

In [4]:
@lru_cache(None)
def fib(n: int) -> int:
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(10)], "| fib(80) =", fib(80), "(không có nhớ sẽ cực chậm)")

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34] | fib(80) = 23416728348467685 (không có nhớ sẽ cực chậm)


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `days = [1,4,6,7,8,20]`, `costs = [2,7,15]`. Chỉ quan tâm các ngày đi.

- `dp[1] = 2` (vé 1 ngày).
- `dp[4]`: các ngày đi `<= 4` là `{1, 4}`: hai vé 1 ngày `4`, hoặc vé 7 ngày phủ cả `1..4` giá `7`: `min(4, 7) = 4`.
- `dp[6]`: `{1, 4, 6}`: ba vé lẻ `6`; hoặc vé 7 ngày (phủ `1..7`) giá `7`: `6`.
- `dp[7]`: `{1, 4, 6, 7}`: một vé 7 ngày giá `7`, hoặc `dp[6] + 2 = 8`: `7`.
- `dp[8]`: `{1,4,6,7,8}`: `dp[7] + 2 = 9`, vé 7 ngày kết thúc ngày 8 (phủ `2..8`) = `dp[1] + 7 = 9`: `9`.
- `dp[20]`: không đi ngày 9..19 nên `dp[19] = dp[8] = 9`; ngày 20: `dp[19] + 2 = 11`: **`11`**.

Cách chia cụ thể: vé 1 ngày (ngày 1) `2`, vé 7 ngày phủ `4..8` (cả `4,6,7,8`) `7`, vé 1 ngày (ngày 20) `2`: `11`.

## 5. Cách 1 — Brute Force: Dijkstra trên các tập ngày đã phủ

**Ý tưởng.** Mỗi vé là một **đoạn liên tiếp** `[s, s + k - 1]` với `s` là ngày bắt đầu bất kỳ (ta thử **mọi** `s` từ `1` đến ngày đi cuối). Vé phủ một **tập con** các ngày đi tàu (biểu diễn bằng bitmask `k` bit). Trạng thái là **tập ngày đã phủ**; từ một trạng thái, mua một vé (loại và ngày bắt đầu bất kỳ) sang trạng thái hợp. Dùng **Dijkstra** (chi phí vé là trọng số) từ tập rỗng đến tập đầy đủ. Không dùng bất kỳ giả định "mua vé đúng ngày phải đi" hay công thức DP nào, nên đây là **oracle độc lập**. Số trạng thái `2^n` nên chỉ dùng cho `n` nhỏ (`<= 10` ngày đi).

Ta viết sẵn dạng tổng quát nhận danh sách độ dài vé `durations` (mặc định `(1, 7, 30)`) để dùng lại ở biến thể.

In [5]:
def mincost_oracle(days: List[int], costs: List[int], durations=(1, 7, 30)) -> int:
    k = len(days)
    full = (1 << k) - 1
    best: Dict[int, int] = {}                     # mask phủ được -> giá nhỏ nhất của một vé phủ đúng mask đó
    for s in range(1, max(days) + 1):             # ngày bắt đầu vé
        for dur, c in zip(durations, costs):
            m = 0
            for i, d in enumerate(days):
                if s <= d < s + dur:
                    m |= 1 << i
            if m and (m not in best or c < best[m]):
                best[m] = c
    dist = {0: 0}
    pq = [(0, 0)]
    while pq:
        d, mask = heapq.heappop(pq)
        if mask == full:
            return d
        if d > dist[mask]:
            continue
        for m, c in best.items():
            nm = mask | m
            if nm not in dist or d + c < dist[nm]:
                dist[nm] = d + c
                heapq.heappush(pq, (d + c, nm))
    raise AssertionError("luôn phủ được bằng vé 1 ngày")

for days, costs, expected in examples[:1]:
    got = mincost_oracle(days, costs)
    print(f"days={days}: oracle = {got}", "OK" if got == expected else "SAI")
    assert got == expected
print("ví dụ 2 có 12 ngày đi: 4096 trạng thái:", mincost_oracle(*examples[1][:2]))
assert mincost_oracle(*examples[1][:2]) == examples[1][2]

days=[1, 4, 6, 7, 8, 20]: oracle = 11 OK
ví dụ 2 có 12 ngày đi: 4096 trạng thái: 17


**Độ phức tạp và nút thắt.** `O(2^n * (số mask khác nhau))`: tăng theo hàm mũ của số ngày đi. Đo thử:

In [6]:
rng = random.Random(983)
for n in (6, 8, 10, 12):
    days = sorted(rng.sample(range(1, 61), n))
    t0 = time.perf_counter()
    mincost_oracle(days, [2, 7, 15])
    print(f"n={n:2} ngày đi: {(time.perf_counter() - t0) * 1000:9.1f} ms")
print("n = 365 -> 2^365 trạng thái: không thể")

n= 6 ngày đi:       0.2 ms
n= 8 ngày đi:       0.7 ms
n=10 ngày đi:       4.0 ms
n=12 ngày đi:      18.8 ms
n = 365 -> 2^365 trạng thái: không thể


Nút thắt: ta xét **mọi tập** ngày đã phủ. Nhưng ngày đi tàu có **thứ tự**: chỉ cần biết "đã lo xong tới ngày nào" (một con số!) vì vé là đoạn liên tiếp. Đó là quy hoạch động.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái là một ngày (hoặc một chỉ số ngày đi), không phải một tập.** Xét các ngày đi từ trái sang phải. Ngày đi **đầu tiên chưa được phủ** phải được phủ bởi một vé; ta có thể giả sử vé đó **bắt đầu đúng ngày này**: mọi ngày trước đó đã được phủ rồi, nên dời ngày bắt đầu của vé về muộn nhất có thể (vẫn phủ ngày này) chỉ làm vé phủ nhiều ngày về sau hơn, không bao giờ kém đi. Mỗi vé sau đó nhảy tới **ngày đi đầu tiên không được phủ**. Do đó:

`f(i) = min` trên ba loại vé của `cost + f(j)`, với `j` là chỉ số ngày đi đầu tiên `>= days[i] + k`.

Có hai cách tính tương đương: theo **chỉ số ngày đi** (Cách 2, 4) hoặc theo **lịch từng ngày** (Cách 3).

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy có nhớ `f(i)` theo chỉ số ngày đi, nhảy bằng `bisect_left` |
| 3 | DP theo lịch: `dp[d]` cho `d = 1..365` (nộp) |
| 4 | DP chỉ trên các ngày đi, chạy ngược từ cuối, tìm vị trí nhảy bằng `bisect_left` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ theo chỉ số ngày đi

**Giải thích từng đoạn.**

- `f(i)`: chi phí nhỏ nhất để đi đủ các ngày `days[i:]`; `f(n) = 0`.
- Với mỗi loại vé độ dài `k` và giá `c`: vé bắt đầu ngày `days[i]` phủ các ngày `< days[i] + k`; ngày đi đầu tiên chưa phủ có chỉ số `j = bisect_left(days, days[i] + k)`.
- `f(i) = min(c + f(j))` trên ba loại vé.

In [7]:
def mincost_memo(days: List[int], costs: List[int]) -> int:
    n = len(days)

    @lru_cache(None)
    def f(i: int) -> int:
        if i == n:
            return 0
        best = float("inf")
        for k, c in zip((1, 7, 30), costs):
            j = bisect_left(days, days[i] + k)
            best = min(best, c + f(j))
        return best

    return f(0)

for days, costs, expected in examples:
    got = mincost_memo(days, costs)
    print(f"days={days}: memo = {got}", "OK" if got == expected else "SAI")
    assert got == expected

days=[1, 4, 6, 7, 8, 20]: memo = 11 OK
days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31]: memo = 17 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        from functools import lru_cache
        from bisect import bisect_left
        n = len(days)

        @lru_cache(None)
        def f(i: int) -> int:
            if i == n:
                return 0
            best = float("inf")
            for k, c in zip((1, 7, 30), costs):
                j = bisect_left(days, days[i] + k)     # ngày đi đầu tiên chưa được vé phủ
                best = min(best, c + f(j))
            return best

        return f(0)

sol_memo = Solution().mincostTickets
for days, costs, expected in examples:
    assert sol_memo(days, costs) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `n` trạng thái, mỗi trạng thái 3 lựa chọn với một `bisect` `O(log n)`: `O(n log n)`; bộ nhớ `O(n)` (bảng nhớ và ngăn xếp đệ quy tối đa `n <= 365`, an toàn với giới hạn đệ quy `1000`).

---

### 7.2. Cách 3 — DP theo lịch từng ngày (cách nộp chính)

**Giải thích từng đoạn.**

- `travel = set(days)`: tra "hôm nay có đi không" trong `O(1)`.
- `dp = [0] * (last + 1)` với `last = days[-1]` (không cần tính quá ngày đi cuối).
- Với `d = 1..last`: nếu `d` không đi thì `dp[d] = dp[d - 1]`; nếu đi thì `dp[d] = min(dp[d-1] + c1, dp[max(0, d-7)] + c7, dp[max(0, d-30)] + c30)`.
- Đáp án `dp[last]`.

In [9]:
def mincost_calendar(days: List[int], costs: List[int]) -> int:
    travel = set(days)
    last = days[-1]
    dp = [0] * (last + 1)
    for d in range(1, last + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
        else:
            dp[d] = min(dp[d - 1] + costs[0],
                        dp[max(0, d - 7)] + costs[1],
                        dp[max(0, d - 30)] + costs[2])
    return dp[last]

for days, costs, expected in examples:
    got = mincost_calendar(days, costs)
    print(f"days={days}: DP lịch = {got}", "OK" if got == expected else "SAI")
    assert got == expected

days=[1, 4, 6, 7, 8, 20]: DP lịch = 11 OK
days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31]: DP lịch = 17 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        travel = set(days)
        last = days[-1]
        dp = [0] * (last + 1)               # dp[d]: chi phí nhỏ nhất để đi đủ mọi ngày đi <= d
        for d in range(1, last + 1):
            if d not in travel:
                dp[d] = dp[d - 1]           # hôm nay không đi: không cần vé mới
            else:
                dp[d] = min(dp[d - 1] + costs[0],              # vé 1 ngày
                            dp[max(0, d - 7)] + costs[1],       # vé 7 ngày kết thúc ở d
                            dp[max(0, d - 30)] + costs[2])      # vé 30 ngày kết thúc ở d
        return dp[last]

sol_calendar = Solution().mincostTickets
for days, costs, expected in examples:
    assert sol_calendar(days, costs) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(D)` với `D <= 365` là ngày đi cuối; bộ nhớ `O(D)`.

---

### 7.3. Cách 4 — DP chỉ trên các ngày đi, chạy ngược

**Ý tưởng khác.** Không cần xét các ngày không đi. Gọi `dp[i]` là chi phí nhỏ nhất để đi đủ `days[i:]`; chạy `i` từ `n - 1` về `0`. Với mỗi loại vé `k, c`: vé bắt đầu ngày `days[i]`, nhảy tới `j = bisect_left(days, days[i] + k)` (có thể dùng con trỏ vì `j` không giảm khi `i` giảm... ngược lại, `j` **không tăng**), `dp[i] = min(c + dp[j])`.

Đây là Cách 2 viết **bottom-up** (không đệ quy), `dp[n] = 0`.

In [11]:
def mincost_days_dp(days: List[int], costs: List[int]) -> int:
    n = len(days)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        dp[i] = min(c + dp[bisect_left(days, days[i] + k)] for k, c in zip((1, 7, 30), costs))
    return dp[0]

for days, costs, expected in examples:
    got = mincost_days_dp(days, costs)
    print(f"days={days}: DP trên ngày đi = {got}", "OK" if got == expected else "SAI")
    assert got == expected

days=[1, 4, 6, 7, 8, 20]: DP trên ngày đi = 11 OK
days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31]: DP trên ngày đi = 17 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        from bisect import bisect_left
        n = len(days)
        dp = [0] * (n + 1)                  # dp[i]: chi phí nhỏ nhất để đi đủ days[i:], dp[n] = 0
        for i in range(n - 1, -1, -1):
            dp[i] = min(c + dp[bisect_left(days, days[i] + k)]
                        for k, c in zip((1, 7, 30), costs))
        return dp[0]

sol_days_dp = Solution().mincostTickets
for days, costs, expected in examples:
    assert sol_days_dp(days, costs) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n log n)` thời gian (mỗi `bisect` `O(log n)`, có thể xuống `O(n)` bằng ba con trỏ), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. DP theo lịch cho ví dụ 1

In `dp[d]` kèm lý do ở từng ngày (đi hay không, ba ứng viên).

In [13]:
def trace_calendar(days, costs):
    travel = set(days)
    last = days[-1]
    dp = [0] * (last + 1)
    for d in range(1, last + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
            print(f"d={d:2}: không đi -> dp = dp[{d - 1}] = {dp[d]}")
        else:
            c1, c7, c30 = dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1], dp[max(0, d - 30)] + costs[2]
            dp[d] = min(c1, c7, c30)
            print(f"d={d:2}: ĐI -> 1 ngày: dp[{d - 1}]+{costs[0]}={c1}; 7 ngày: dp[{max(0, d - 7)}]+{costs[1]}={c7}; 30 ngày: dp[{max(0, d - 30)}]+{costs[2]}={c30} -> dp = {dp[d]}")
    return dp[last]

assert trace_calendar([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11

d= 1: ĐI -> 1 ngày: dp[0]+2=2; 7 ngày: dp[0]+7=7; 30 ngày: dp[0]+15=15 -> dp = 2
d= 2: không đi -> dp = dp[1] = 2
d= 3: không đi -> dp = dp[2] = 2
d= 4: ĐI -> 1 ngày: dp[3]+2=4; 7 ngày: dp[0]+7=7; 30 ngày: dp[0]+15=15 -> dp = 4
d= 5: không đi -> dp = dp[4] = 4
d= 6: ĐI -> 1 ngày: dp[5]+2=6; 7 ngày: dp[0]+7=7; 30 ngày: dp[0]+15=15 -> dp = 6
d= 7: ĐI -> 1 ngày: dp[6]+2=8; 7 ngày: dp[0]+7=7; 30 ngày: dp[0]+15=15 -> dp = 7
d= 8: ĐI -> 1 ngày: dp[7]+2=9; 7 ngày: dp[1]+7=9; 30 ngày: dp[0]+15=15 -> dp = 9
d= 9: không đi -> dp = dp[8] = 9
d=10: không đi -> dp = dp[9] = 9
d=11: không đi -> dp = dp[10] = 9
d=12: không đi -> dp = dp[11] = 9
d=13: không đi -> dp = dp[12] = 9
d=14: không đi -> dp = dp[13] = 9
d=15: không đi -> dp = dp[14] = 9
d=16: không đi -> dp = dp[15] = 9
d=17: không đi -> dp = dp[16] = 9
d=18: không đi -> dp = dp[17] = 9
d=19: không đi -> dp = dp[18] = 9
d=20: ĐI -> 1 ngày: dp[19]+2=11; 7 ngày: dp[13]+7=16; 30 ngày: dp[0]+15=15 -> dp = 11


### 8.2. Đệ quy có nhớ cho ví dụ 1

In từng lời gọi `f(i)`: các lựa chọn và chỉ số nhảy `j`.

In [14]:
def trace_memo(days, costs):
    n = len(days)
    memo = {}

    def f(i, depth=0):
        pad = "  " * depth
        if i == n:
            return 0
        if i in memo:
            print(f"{pad}f({i}) đã tính = {memo[i]}")
            return memo[i]
        print(f"{pad}f({i}): ngày {days[i]}")
        best = float("inf")
        for k, c in zip((1, 7, 30), costs):
            j = bisect_left(days, days[i] + k)
            val = c + f(j, depth + 1)
            print(f"{pad}  vé {k:2} ngày (giá {c}) -> nhảy tới j={j}: tổng {val}")
            best = min(best, val)
        memo[i] = best
        print(f"{pad}f({i}) = {best}")
        return best

    return f(0)

assert trace_memo([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11

f(0): ngày 1
  f(1): ngày 4
    f(2): ngày 6
      f(3): ngày 7
        f(4): ngày 8
          f(5): ngày 20
            vé  1 ngày (giá 2) -> nhảy tới j=6: tổng 2
            vé  7 ngày (giá 7) -> nhảy tới j=6: tổng 7
            vé 30 ngày (giá 15) -> nhảy tới j=6: tổng 15
          f(5) = 2
          vé  1 ngày (giá 2) -> nhảy tới j=5: tổng 4
          f(5) đã tính = 2
          vé  7 ngày (giá 7) -> nhảy tới j=5: tổng 9
          vé 30 ngày (giá 15) -> nhảy tới j=6: tổng 15
        f(4) = 4
        vé  1 ngày (giá 2) -> nhảy tới j=4: tổng 6
        f(5) đã tính = 2
        vé  7 ngày (giá 7) -> nhảy tới j=5: tổng 9
        vé 30 ngày (giá 15) -> nhảy tới j=6: tổng 15
      f(3) = 6
      vé  1 ngày (giá 2) -> nhảy tới j=3: tổng 8
      f(5) đã tính = 2
      vé  7 ngày (giá 7) -> nhảy tới j=5: tổng 9
      vé 30 ngày (giá 15) -> nhảy tới j=6: tổng 15
    f(2) = 8
    vé  1 ngày (giá 2) -> nhảy tới j=2: tổng 10
    f(5) đã tính = 2
    vé  7 ngày (giá 7) -> nhảy tới j=5: tổng 9
    

### 8.3. Vì sao tham lam theo giá mỗi ngày không đủ

Với `costs = [2, 7, 15]` vé 7 ngày giá mỗi ngày là `1` (rẻ hơn vé 1 ngày giá `2`), nhưng nếu chỉ cần đi 2 ngày thì vé 1 ngày + 1 ngày (`4`) rẻ hơn vé 7 ngày (`7`). Đáp án phụ thuộc **cách các ngày dồn lại**, nên cần DP. Ta sẽ dựng phản ví dụ cụ thể bằng tìm kiếm ở mục 12.

## 9. Code nộp LeetCode

Ta chọn **Cách 3** (DP theo lịch): ngắn, `O(365)`, dễ chứng minh đúng.

```python
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        travel = set(days)
        last = days[-1]
        dp = [0] * (last + 1)               # dp[d]: chi phí nhỏ nhất để đi đủ mọi ngày đi <= d
        for d in range(1, last + 1):
            if d not in travel:
                dp[d] = dp[d - 1]           # hôm nay không đi: không cần vé mới
            else:
                dp[d] = min(dp[d - 1] + costs[0],              # vé 1 ngày
                            dp[max(0, d - 7)] + costs[1],       # vé 7 ngày kết thúc ở d
                            dp[max(0, d - 30)] + costs[2])      # vé 30 ngày kết thúc ở d
        return dp[last]
```

In [15]:
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        travel = set(days)
        last = days[-1]
        dp = [0] * (last + 1)               # dp[d]: chi phí nhỏ nhất để đi đủ mọi ngày đi <= d
        for d in range(1, last + 1):
            if d not in travel:
                dp[d] = dp[d - 1]           # hôm nay không đi: không cần vé mới
            else:
                dp[d] = min(dp[d - 1] + costs[0],              # vé 1 ngày
                            dp[max(0, d - 7)] + costs[1],       # vé 7 ngày kết thúc ở d
                            dp[max(0, d - 30)] + costs[2])      # vé 30 ngày kết thúc ở d
        return dp[last]

sol = Solution()
for days, costs, expected in examples:
    got = sol.mincostTickets(days, costs)
    print(f"days={days}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

days=[1, 4, 6, 7, 8, 20]: 11 OK
days=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31]: 17 OK


## 10. Vì sao đúng

**Định nghĩa.** `dp[d]` = chi phí nhỏ nhất của một tập vé phủ **mọi ngày đi tàu `<= d`** (các vé có thể kéo dài quá `d`; ta chỉ đòi các ngày đi `<= d` được phủ).

**Ngày không đi.** Nếu `d` không phải ngày đi, tập ngày đi `<= d` giống hệt tập `<= d - 1`, nên `dp[d] = dp[d - 1]`.

**Ngày đi.** Xét một phương án tối ưu cho các ngày đi `<= d`. Ngày `d` được phủ bởi **một** vé nào đó. Nếu vé 1 ngày: phần còn lại phủ các ngày đi `<= d - 1` nên tốn `>= dp[d - 1]`: tổng `>= dp[d-1] + c1`. Nếu vé có độ dài `k` (`7` hoặc `30`) phủ ngày `d`, ta có thể **dời vé cho nó kết thúc đúng ở ngày `d`** (phủ `d-k+1..d`): vé mới vẫn phủ ngày `d` và mọi ngày đi `<= d` mà vé cũ từng phủ... (vé cũ phủ `s..s+k-1` với `s <= d <= s+k-1`; các ngày đi trong khoảng đó **không vượt `d`** nằm trong `s..d`, mà `s >= d-k+1`, nên vẫn thuộc `d-k+1..d`). Các ngày đi `<= d - k` còn lại cần được phủ bởi các vé còn lại: tốn `>= dp[d - k]`. Vậy `dp[d] >= min(dp[d-1] + c1, dp[d-7] + c7, dp[d-30] + c30)`. Ngược lại, mỗi lựa chọn trong `min` cho một phương án hợp lệ (vé mới ghép với phương án tối ưu của `dp[...]`), nên `dp[d] <=` nó. Chỉ số `d - k < 0` nghĩa là không còn ngày đi nào cần phủ trước đó: chi phí `0`, đó là lý do dùng `max(0, ...)`.

**Đáp án.** `dp[365]` (hoặc `dp[days[-1]]`) vì sau ngày đi cuối không còn ngày nào cần phủ.

**Cách 2 và 4** tính cùng giá trị theo chỉ số ngày đi: vé bắt đầu đúng ngày đi đầu tiên chưa phủ (tối ưu khi dời vé bắt đầu muộn nhất có thể vẫn phủ ngày đó: vé phủ thêm về sau, không kém đi), rồi nhảy tới ngày đi đầu tiên chưa phủ.

**Kiểm chứng bằng máy.** Oracle (Cách 1) **không** dùng giả thiết nào của DP: thử mọi ngày bắt đầu vé và dùng Dijkstra trên tập ngày đi đã phủ. Ta so sánh mọi cách với oracle trên rất nhiều ca ngẫu nhiên có seed (bộ giá bất kỳ, kể cả vé dài **rẻ hơn** vé ngắn), rồi các cách nhanh với nhau trên ca lớn (cả 365 ngày).

In [16]:
def gen_small(rng, max_day=45, max_n=10):
    n = rng.randint(1, max_n)
    days = sorted(rng.sample(range(1, max_day + 1), n))
    costs = [rng.randint(1, 20) for _ in range(3)]
    return days, costs

rng = random.Random(1983)
for _ in range(300):
    days, costs = gen_small(rng)
    ref = mincost_oracle(days, costs)
    assert mincost_memo(days, costs) == mincost_calendar(days, costs) == mincost_days_dp(days, costs) == sol.mincostTickets(days, costs) == ref, (days, costs, ref)
print("300 ca ngẫu nhiên nhỏ: các cách khớp oracle Dijkstra")

300 ca ngẫu nhiên nhỏ: các cách khớp oracle Dijkstra


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dijkstra trên tập ngày | `O(2^n * m)` | `O(2^n)` |
| 2. Đệ quy có nhớ theo ngày đi | `O(n log n)` | `O(n)` |
| 3. DP theo lịch | `O(D)`, `D <= 365` | `O(D)` |
| 4. DP trên ngày đi | `O(n log n)` | `O(n)` |

Đo ở cỡ tối đa (`n = 365`, tức đi **mọi** ngày trong năm) và ngày đi thưa:

In [17]:
all_days = list(range(1, 366))
sparse = sorted(random.Random(11).sample(range(1, 366), 100))
for label, days in (("365 ngày liên tục", all_days), ("100 ngày thưa", sparse)):
    ref = mincost_calendar(days, [2, 7, 15])
    for name, fn in (("Cách 2", sol_memo), ("Cách 3", sol_calendar), ("Cách 4", sol_days_dp), ("nộp", sol.mincostTickets)):
        t0 = time.perf_counter()
        for _ in range(100):
            got = fn(days, [2, 7, 15])
        dt = (time.perf_counter() - t0) / 100 * 1000
        assert got == ref, name
        print(f"{label:18} {name:7}: {dt:7.3f} ms, đáp án {got}")

365 ngày liên tục  Cách 2 :   0.416 ms, đáp án 187


365 ngày liên tục  Cách 3 :   0.074 ms, đáp án 187


365 ngày liên tục  Cách 4 :   0.331 ms, đáp án 187
365 ngày liên tục  nộp    :   0.068 ms, đáp án 187
100 ngày thưa      Cách 2 :   0.093 ms, đáp án 158
100 ngày thưa      Cách 3 :   0.032 ms, đáp án 158


100 ngày thưa      Cách 4 :   0.086 ms, đáp án 158
100 ngày thưa      nộp    :   0.032 ms, đáp án 158


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu với oracle Dijkstra, test ngẫu nhiên có seed (đúng ràng buộc, cả ca lớn) và các tính chất.

In [18]:
def all_solutions():
    return {"memo": mincost_memo, "calendar": mincost_calendar, "days_dp": mincost_days_dp,
            "sol_memo": sol_memo, "sol_calendar": sol_calendar, "sol_days_dp": sol_days_dp, "final": sol.mincostTickets}

def check_all(days, costs, expected=None, brute=True):
    results = {name: fn(list(days), list(costs)) for name, fn in all_solutions().items()}
    ref = results["calendar"]
    assert len(set(results.values())) == 1, results
    if brute and len(days) <= 10 and days[-1] <= 80:
        assert mincost_oracle(days, costs) == ref, (days, costs, mincost_oracle(days, costs), ref)
    if expected is not None:
        assert ref == expected, (days, costs, ref, expected)
    return ref

# 12.1 ví dụ đề
for days, costs, expected in examples:
    check_all(days, costs, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], [2, 7, 15], 2),                                     # một ngày: vé 1 ngày
    ([365], [2, 7, 15], 2),                                   # ngày cuối năm
    ([1], [5, 3, 4], 3),                                      # vé dài rẻ hơn vé 1 ngày: chọn vé 7 ngày giá 3
    ([1, 2], [2, 7, 15], 4),                                  # hai ngày: 2 + 2 < 7
    ([1, 2, 3, 4, 5, 6, 7], [2, 7, 15], 7),                   # đi 7 ngày liên tiếp: vé 7 ngày giá 7 < 14
    ([1, 8], [2, 7, 15], 4),                                  # hai ngày cách nhau 7: hai vé 1 ngày
    ([1, 7], [2, 7, 15], 4),                                  # vé 7 ngày phủ 1..7 nhưng 2+2=4 rẻ hơn 7
    ([1, 2, 3, 4], [2, 7, 15], 7),                            # 4 ngày: vé 7 ngày giá 7 < 8
    ([1, 30], [2, 7, 15], 4),                                 # cách xa nhau: hai vé 1 ngày
    (list(range(1, 31)), [2, 7, 15], 15),                     # 30 ngày liên tục: một vé 30 ngày
    (list(range(1, 366)), [1, 1, 1], 13),                     # mọi vé giá 1, đi cả năm: 365 / 30 = 12.17 -> 13 vé 30 ngày
    ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [1000, 1000, 1000], 2000),   # giá lớn: cần 2 vé (30 ngày + 30 ngày) hoặc 7 ngày + ...
    ([1, 100, 200, 300], [3, 100, 100], 12),                  # xa nhau: bốn vé 1 ngày
    ([2, 3, 4, 5, 6, 7, 8, 15], [3, 10, 20], 13),             # tính tay ở bài tập 2
]
for days, costs, expected in edge:
    if expected is None:
        continue
    check_all(days, costs, expected, brute=(len(days) <= 10 and days[-1] <= 80))
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle)
def wrong_negative_index(days, costs):             # dp[d - 7], dp[d - 30] không chặn: chỉ số âm lấy phần tử cuối
    travel = set(days)
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
        else:
            dp[d] = min(dp[d - 1] + costs[0], dp[d - 7] + costs[1], dp[d - 30] + costs[2])
    return dp[days[-1]]

def wrong_no_carry(days, costs):                   # quên dp[d] = dp[d - 1] ở ngày không đi
    travel = set(days)
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        if d in travel:
            dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1], dp[max(0, d - 30)] + costs[2])
    return dp[days[-1]]

def wrong_greedy(days, costs):                     # tham lam: tại ngày đi chưa phủ, chọn vé rẻ nhất theo giá mỗi NGÀY ĐI được phủ
    total, i, n = 0, 0, len(days)
    while i < n:
        best = None
        for k, c in zip((1, 7, 30), costs):
            j = bisect_left(days, days[i] + k)
            ratio = c / (j - i)
            if best is None or ratio < best[0]:
                best = (ratio, c, j)
        total += best[1]
        i = best[2]
    return total

def wrong_off_by_one(days, costs):                 # lệch 1: vé 7 ngày coi như phủ 6 ngày (dp[d - 6])
    travel = set(days)
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
        else:
            dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 6)] + costs[1], dp[max(0, d - 29)] + costs[2])
    return dp[days[-1]]

def wrong_no_30(days, costs):                      # bỏ sót vé 30 ngày
    travel = set(days)
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
        else:
            dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1])
    return dp[days[-1]]

def find_counter(wrong, seed=9, tries=30000):
    rng = random.Random(seed)
    for _ in range(tries):
        days, costs = gen_small(rng, max_day=40, max_n=8)
        try:
            got = wrong(list(days), list(costs))
        except Exception:
            continue                                  # chỉ lấy ca cho ra đáp án SAI (không phải ca ném lỗi)
        if got != mincost_calendar(days, costs):
            return days, costs
    return None

counters = {}
for name, fn in (("chỉ số âm dp[d - 7]", wrong_negative_index), ("quên dp[d] = dp[d - 1] ngày không đi", wrong_no_carry),
                 ("tham lam theo giá mỗi ngày", wrong_greedy), ("lệch một đơn vị độ dài vé", wrong_off_by_one),
                 ("bỏ sót vé 30 ngày", wrong_no_30)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    days, costs = case
    print(f"{name:38} days={days} costs={costs}: sai -> {fn(list(days), list(costs))}, đúng -> {mincost_calendar(days, costs)}")
    check_all(days, costs)
print("12.3 phản ví dụ: OK")

chỉ số âm dp[d - 7]                    days=[7, 25] costs=[10, 7, 8]: sai -> 14, đúng -> 8
quên dp[d] = dp[d - 1] ngày không đi   days=[1, 9, 12, 18, 22, 24, 33, 40] costs=[15, 20, 3]: sai -> 3, đúng -> 6
tham lam theo giá mỗi ngày             days=[5, 20, 34, 35, 37] costs=[3, 8, 5]: sai -> 10, đúng -> 8
lệch một đơn vị độ dài vé              days=[1, 2, 5, 6, 21, 22, 29, 30] costs=[19, 5, 13]: sai -> 15, đúng -> 13
bỏ sót vé 30 ngày                      days=[1, 9, 12, 18, 22, 24, 33, 40] costs=[15, 20, 3]: sai -> 85, đúng -> 6
12.3 phản ví dụ: OK


In [20]:
# 12.4 ngẫu nhiên nhỏ so với oracle Dijkstra: nhiều dạng giá (kể cả vé dài rẻ hơn vé ngắn) và mật độ ngày
rng = random.Random(4983)
for _ in range(500):
    days, costs = gen_small(rng, max_day=rng.choice([10, 20, 45, 70]), max_n=rng.randint(1, 10))
    if rng.random() < 0.3:
        costs = sorted(costs)                      # giá tăng theo độ dài (thường gặp)
    check_all(days, costs)
print("12.4 ngẫu nhiên nhỏ so với oracle (500 ca): OK")

# 12.5 ngẫu nhiên đúng ràng buộc (days tới 365, giá tới 1000): các cách đối chiếu nhau
rng = random.Random(5983)
for _ in range(500):
    n = rng.randint(1, 365)
    days = sorted(rng.sample(range(1, 366), n))
    costs = [rng.randint(1, 1000) for _ in range(3)]
    if rng.random() < 0.5:
        costs = sorted(costs)
    check_all(days, costs, brute=False)
print("12.5 ngẫu nhiên cỡ lớn (500 ca): OK")

# 12.6 tính chất: chặn trên/dưới, đơn điệu
rng = random.Random(6983)
for _ in range(300):
    n = rng.randint(1, 120)
    days = sorted(rng.sample(range(1, 366), n))
    costs = [rng.randint(1, 1000) for _ in range(3)]
    ans = sol.mincostTickets(days, costs)
    assert ans <= n * costs[0]                                         # luôn có thể mua vé 1 ngày mỗi ngày
    assert ans <= -(-days[-1] // 30) * costs[2] + 0                    # phủ [1, days[-1]] bằng các vé 30 ngày
    assert ans >= min(costs)                                           # ít nhất một vé
    sub = days[:max(1, n // 2)]
    assert sol.mincostTickets(sub, costs) <= ans                       # bớt ngày đi (tiền tố) không tốn thêm
    assert sol.mincostTickets(days, [c + 1 for c in costs]) >= ans     # tăng giá không làm giảm chi phí
    assert sol.mincostTickets(days, [2 * c for c in costs]) == 2 * ans # nhân đôi giá nhân đôi đáp án
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ so với oracle (500 ca): OK


12.5 ngẫu nhiên cỡ lớn (500 ca): OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ số âm: `dp[d - 7]` khi `d < 7`

Trong Python `dp[d - 7]` với `d < 7` không báo lỗi mà lấy phần tử **tính từ cuối**. Kết quả có thể tình cờ đúng hoặc sai tùy dữ liệu.

In [21]:
days, costs = counters["chỉ số âm dp[d - 7]"]
print("days =", days, "| costs =", costs, "| sai ->", wrong_negative_index(days, costs), "| đúng ->", Solution().mincostTickets(days, costs))
assert wrong_negative_index(days, costs) != Solution().mincostTickets(days, costs)

days = [7, 25] | costs = [10, 7, 8] | sai -> 14 | đúng -> 8


Vì sao sai: `dp[-k]` là phần tử thứ `k` từ cuối, không phải "chưa có ngày nào"; khi `d - 30 < -len(dp)` còn có thể ném `IndexError`. (Nhiều lời giải dùng `dp = [0] * 366` nên các chỉ số âm trỏ vào phần chưa tính, toàn `0`: đúng một cách **tình cờ**.) **Đúng:** `dp[max(0, d - 7)]`.

### Lỗi 2 — Quên `dp[d] = dp[d - 1]` ở ngày không đi tàu

Nếu chỉ gán `dp[d]` ở ngày đi, các ngày không đi giữ giá trị `0` và mất chi phí đã tích lũy.

In [22]:
days, costs = counters["quên dp[d] = dp[d - 1] ngày không đi"]
print("days =", days, "| costs =", costs, "| sai ->", wrong_no_carry(days, costs), "| đúng ->", Solution().mincostTickets(days, costs))
assert wrong_no_carry(days, costs) != Solution().mincostTickets(days, costs)

days = [1, 9, 12, 18, 22, 24, 33, 40] | costs = [15, 20, 3] | sai -> 3 | đúng -> 6


Vì sao sai: `dp[d]` phải là chi phí cho **mọi ngày đi `<= d`**, kể cả khi `d` không đi. **Đúng:** `dp[d] = dp[d - 1]`.

### Lỗi 3 — Tham lam theo giá mỗi ngày

Chọn vé có giá/ngày thấp nhất ở mỗi bước không đúng, vì số ngày **thực sự đi** trong vé mới là thứ được hưởng lợi, và các lựa chọn sau phụ thuộc vào đó.

In [23]:
days, costs = counters["tham lam theo giá mỗi ngày"]
print("days =", days, "| costs =", costs, "| sai ->", wrong_greedy(days, costs), "| đúng ->", Solution().mincostTickets(days, costs))
assert wrong_greedy(days, costs) != Solution().mincostTickets(days, costs)

days = [5, 20, 34, 35, 37] | costs = [3, 8, 5] | sai -> 10 | đúng -> 8


Vì sao sai: tối ưu cục bộ không bảo đảm tối ưu toàn cục khi các vé chồng lấn và ảnh hưởng ngày tiếp theo. **Đúng:** quy hoạch động.

### Lỗi 4 — Lệch một đơn vị độ dài vé

Vé 7 ngày mua vào ngày `d` phủ `d..d+6`, nên vé kết thúc ở ngày `d` bắt đầu từ `d - 6` và phần trước đó là `dp[d - 7]` (không phải `dp[d - 6]`).

In [24]:
days, costs = counters["lệch một đơn vị độ dài vé"]
print("days =", days, "| costs =", costs, "| sai ->", wrong_off_by_one(days, costs), "| đúng ->", Solution().mincostTickets(days, costs))
assert wrong_off_by_one(days, costs) != Solution().mincostTickets(days, costs)

days = [1, 2, 5, 6, 21, 22, 29, 30] | costs = [19, 5, 13] | sai -> 15 | đúng -> 13


Vì sao sai: `dp[d - 6]` coi như vé chỉ phủ 6 ngày. **Đúng:** `dp[d - 7]` cho vé 7 ngày, `dp[d - 30]` cho vé 30 ngày.

### Lỗi 5 — Bỏ sót vé 30 ngày

Hai loại vé đầu đủ cho ví dụ 1, và ví dụ 2 với giá `[2, 7, 15]` cũng chưa lộ lỗi (hai phương án cùng `17`); lỗi lộ ra khi vé 30 ngày rẻ hơn hẳn.

In [25]:
days, costs = counters["bỏ sót vé 30 ngày"]
print("days =", days, "| costs =", costs, "| sai ->", wrong_no_30(days, costs), "| đúng ->", Solution().mincostTickets(days, costs))
assert wrong_no_30(days, costs) != Solution().mincostTickets(days, costs)
print("ví dụ 2 của đề với giá [2, 7, 15] vẫn đúng một cách tình cờ:", wrong_no_30(*examples[1][:2]), "= 17; cần giá vé 30 ngày rẻ hơn hẳn để lộ lỗi")

days = [1, 9, 12, 18, 22, 24, 33, 40] | costs = [15, 20, 3] | sai -> 85 | đúng -> 6
ví dụ 2 của đề với giá [2, 7, 15] vẫn đúng một cách tình cờ: 17 = 17; cần giá vé 30 ngày rẻ hơn hẳn để lộ lỗi


Vì sao sai: khi đi nhiều ngày liên tục, vé 30 ngày có thể rẻ hơn nhiều vé 7 ngày. **Đúng:** xét cả ba loại vé ở mỗi ngày đi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "phủ các ngày/vị trí bằng đoạn liên tiếp có nhiều độ dài và giá khác nhau, tối thiểu chi phí" ⇒ **DP theo trục thời gian**: `dp[i]` = chi phí tối thiểu cho mọi thứ `<= i`.
- Ngày không cần phủ thì `dp[i] = dp[i - 1]`; ngày cần phủ thì thử từng loại đoạn kết thúc ở `i`.
- Chặn chỉ số âm bằng `max(0, ...)`, hoặc dịch mảng `dp` thêm phần đệm (`dp` có độ dài `D + 31`).
- Giá **không** nhất thiết tăng theo độ dài: không bao giờ giả định vé dài đắt hơn.
- Hai cách viết tương đương: **theo lịch** (`O(365)`) và **theo chỉ số ngày đi** (`O(n log n)`).
- Cùng họ: Coin Change (322), Min Cost Climbing Stairs (746), Paint Fence, Minimum Cost to Cut a Stick (1547).
- Test nên có: một ngày, ngày `365`, đi cả năm, ngày cách xa, vé dài rẻ hơn vé ngắn, giá bằng nhau.

## 15. Biến thể

1. **Số loại vé và độ dài tùy ý:** `dp[d] = min(dp[max(0, d - dur)] + c for dur, c in tickets)`; bài tập 5.
2. **Đếm số cách mua vé tối ưu** hoặc **truy vết** danh sách vé đã mua.
3. **Ngày đi không giới hạn 365** (rất thưa): DP theo chỉ số ngày đi (Cách 4), không theo lịch.
4. Bài liên quan: 322, 746, 1547.

Biến thể 1 (độ dài và giá tùy ý) kiểm bằng oracle Dijkstra (đã viết sẵn ở dạng tổng quát):

In [26]:
def mincost_general(days: List[int], durations: List[int], costs: List[int]) -> int:
    n = len(days)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        dp[i] = min(c + dp[bisect_left(days, days[i] + dur)] for dur, c in zip(durations, costs))
    return dp[0]

assert mincost_general([1, 4, 6, 7, 8, 20], [1, 7, 30], [2, 7, 15]) == 11
rng = random.Random(15)
for _ in range(300):
    days = sorted(rng.sample(range(1, 40), rng.randint(1, 9)))
    m = rng.randint(1, 4)
    durations = [rng.randint(1, 15) for _ in range(m)]
    costs = [rng.randint(1, 15) for _ in range(m)]
    assert mincost_general(days, durations, costs) == mincost_oracle(days, costs, durations), (days, durations, costs)
print("biến thể độ dài/giá tùy ý: khớp oracle trên 300 ca")

biến thể độ dài/giá tùy ý: khớp oracle trên 300 ca


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dijkstra trên tập ngày | Mọi tập đã phủ | `O(2^n * m)` | `O(2^n)` | Oracle, `n <= 10` |
| 2. Đệ quy có nhớ | `f(i)` theo ngày đi | `O(n log n)` | `O(n)` | Dễ suy ra từ định nghĩa |
| 3. DP theo lịch | `dp[d]` từng ngày | `O(365)` | `O(365)` | **Cách nộp chính** |
| 4. DP trên ngày đi | Bottom-up + `bisect` | `O(n log n)` | `O(n)` | Ngày đi thưa/ngày lớn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(max(0, 5 - 7), [0, 1, 2, 3][5 - 7])
print(bisect_left([1, 4, 6, 7, 8, 20], 9), bisect_left([1, 4, 6, 7, 8, 20], 5))
print(Solution().mincostTickets([1, 2, 3], [2, 7, 15]))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai số của dòng 1
my_guess_2 = None      # hai số của dòng 2
my_guess_3 = None      # dòng 3

### Bài 2 — Tính tay (dễ-vừa)

`days = [2, 3, 4, 5, 6, 7, 8, 15]`, `costs = [3, 10, 20]`. Tính chi phí nhỏ nhất.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DP theo lịch.

In [29]:
def exercise_3(days: List[int], costs: List[int]) -> int:
    travel = set(days)
    last = days[-1]
    dp = [0] * (last + 1)
    for d in range(1, last + 1):
        # TODO 1: nếu d không đi thì dp[d] = dp[d - 1]
        # TODO 2: nếu d đi thì dp[d] = min của ba loại vé (nhớ chặn chỉ số âm bằng max(0, ...))
        pass
    return dp[last]


# for days, costs, expected in examples:
#     assert exercise_3(days, costs) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(days, costs)` bằng **đệ quy có nhớ** theo chỉ số ngày đi (dùng `lru_cache` và `bisect_left`).

In [30]:
def exercise_4(days: List[int], costs: List[int]) -> int:
    # TODO
    pass


# for days, costs, expected in examples:
#     assert exercise_4(days, costs) == expected

### Bài 5 — Mini challenge: số loại vé và độ dài tùy ý (khó)

Viết `exercise_5(days, durations, costs)` trả về chi phí nhỏ nhất khi có `m` loại vé, vé loại `t` phủ `durations[t]` ngày liên tiếp và giá `costs[t]` (`days` tăng ngặt, mọi `durations[t] >= 1`).

```python
exercise_5([1, 4, 6, 7, 8, 20], [1, 7, 30], [2, 7, 15]) -> 11
exercise_5([1, 2, 3], [1, 2], [5, 6])                    -> 11   # vé 2 ngày (6) + vé 1 ngày (5)
exercise_5([1, 10], [5], [4])                            -> 8
exercise_5([1, 2, 3, 4, 5], [3, 5], [4, 9])              -> 8    # hai vé 3 ngày (8) hoặc ... xem hàm kiểm tra
```
Gợi ý: Cách 4 với danh sách các loại vé.

In [31]:
def exercise_5(days: List[int], durations: List[int], costs: List[int]) -> int:
    # TODO
    pass


# assert exercise_5([1, 4, 6, 7, 8, 20], [1, 7, 30], [2, 7, 15]) == 11
# assert exercise_5([1, 2, 3], [1, 2], [5, 6]) == 11
# assert exercise_5([1, 10], [5], [4]) == 8
# assert exercise_5([1, 2, 3, 4, 5], [3, 5], [4, 9]) == 8

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `max(0, x)` chặn số âm; chỉ số âm trong danh sách tính từ cuối.
- Mức 2: `[0, 1, 2, 3][-2]` là `2`; `bisect_left` đếm số phần tử nhỏ hơn giá trị.
- Mức 3: dòng 1: `0 2`; dòng 2: `5 2`; dòng 3: `6` (ba vé 1 ngày giá `6` rẻ hơn vé 7 ngày giá `7`).

**Bài 2**
- Mức 1: ba lựa chọn cho nhóm `2..8`: bảy vé 1 ngày (`21`), một vé 7 ngày (`10`).
- Mức 2: ngày `15` cô lập: vé 1 ngày (`3`).
- Mức 3: `10 + 3 = 13`.

**Bài 3**
- Mức 1: `if d not in travel: dp[d] = dp[d - 1]`.
- Mức 2: `else: dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1], dp[max(0, d - 30)] + costs[2])`.
- Mức 3: trả `dp[last]`.

**Bài 4**
- Mức 1: `f(i)` = chi phí để đi đủ `days[i:]`, `f(n) = 0`.
- Mức 2: với mỗi `(k, c)`: `j = bisect_left(days, days[i] + k)`; `c + f(j)`.
- Mức 3: `@lru_cache(None)` rồi `return f(0)`.

**Bài 5**
- Mức 1: giống Cách 4 nhưng duyệt các `dur` bất kỳ.
- Mức 2: `dp[i] = min(c + dp[bisect_left(days, days[i] + dur)] for dur, c in zip(durations, costs))`.
- Mức 3: ví dụ cuối: `[1,2,3,4,5]` với vé 3 ngày giá `4` và vé 5 ngày giá `9`: hai vé 3 ngày (`8`) rẻ hơn một vé 5 ngày (`9`).

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(max(0, 5 - 7), [0, 1, 2, 3][5 - 7])
print(bisect_left([1, 4, 6, 7, 8, 20], 9), bisect_left([1, 4, 6, 7, 8, 20], 5))
print(Solution().mincostTickets([1, 2, 3], [2, 7, 15]))

assert (max(0, 5 - 7), [0, 1, 2, 3][5 - 7]) == (0, 2)
assert (bisect_left([1, 4, 6, 7, 8, 20], 9), bisect_left([1, 4, 6, 7, 8, 20], 5)) == (5, 2)
assert Solution().mincostTickets([1, 2, 3], [2, 7, 15]) == 6

0 2
5 2
6


### Đáp án Bài 2

Nhóm `2..8` (7 ngày liên tục) mua một vé 7 ngày giá `10` (rẻ hơn 7 vé 1 ngày giá `21`); ngày `15` mua vé 1 ngày giá `3`. Tổng `13`.

In [33]:
hand_answer_solution = 13
assert Solution().mincostTickets([2, 3, 4, 5, 6, 7, 8, 15], [3, 10, 20]) == hand_answer_solution
assert mincost_oracle([2, 3, 4, 5, 6, 7, 8, 15], [3, 10, 20]) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_solution(days: List[int], costs: List[int]) -> int:
    travel = set(days)
    last = days[-1]
    dp = [0] * (last + 1)
    for d in range(1, last + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
        else:
            dp[d] = min(dp[d - 1] + costs[0], dp[max(0, d - 7)] + costs[1], dp[max(0, d - 30)] + costs[2])
    return dp[last]

for days, costs, expected in examples:
    assert exercise_3_solution(days, costs) == expected
rng = random.Random(3)
for _ in range(200):
    days, costs = gen_small(rng)
    assert exercise_3_solution(days, costs) == mincost_oracle(days, costs)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_solution(days: List[int], costs: List[int]) -> int:
    n = len(days)

    @lru_cache(None)
    def f(i: int) -> int:
        if i == n:
            return 0
        return min(c + f(bisect_left(days, days[i] + k)) for k, c in zip((1, 7, 30), costs))

    return f(0)

for days, costs, expected in examples:
    assert exercise_4_solution(days, costs) == expected
rng = random.Random(4)
for _ in range(200):
    days, costs = gen_small(rng)
    assert exercise_4_solution(days, costs) == mincost_oracle(days, costs)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_solution(days: List[int], durations: List[int], costs: List[int]) -> int:
    n = len(days)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        dp[i] = min(c + dp[bisect_left(days, days[i] + dur)] for dur, c in zip(durations, costs))
    return dp[0]

assert exercise_5_solution([1, 4, 6, 7, 8, 20], [1, 7, 30], [2, 7, 15]) == 11
assert exercise_5_solution([1, 2, 3], [1, 2], [5, 6]) == 11
assert exercise_5_solution([1, 10], [5], [4]) == 8
assert exercise_5_solution([1, 2, 3, 4, 5], [3, 5], [4, 9]) == 8
rng = random.Random(5)
for _ in range(200):
    days = sorted(rng.sample(range(1, 40), rng.randint(1, 9)))
    m = rng.randint(1, 4)
    durations = [rng.randint(1, 15) for _ in range(m)]
    costs = [rng.randint(1, 15) for _ in range(m)]
    assert exercise_5_solution(days, durations, costs) == mincost_oracle(days, costs, durations)
print("Bài 5 OK")

Bài 5 OK

## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DP theo lịch)

```python
class Solution:
    def mincostTickets(self, days: List[int], costs: List[int]) -> int:
        travel = set(days)
        last = days[-1]
        dp = [0] * (last + 1)               # dp[d]: chi phí nhỏ nhất cho mọi ngày đi <= d
        for d in range(1, last + 1):
            if d not in travel:
                dp[d] = dp[d - 1]           # không đi: không cần vé mới
            else:
                dp[d] = min(dp[d - 1] + costs[0],              # vé 1 ngày
                            dp[max(0, d - 7)] + costs[1],       # vé 7 ngày (chặn chỉ số âm)
                            dp[max(0, d - 30)] + costs[2])      # vé 30 ngày
        return dp[last]
```

Công thức nhớ nhanh: `dp[d] = dp[d - 1]` nếu không đi; ngược lại `min(dp[d - k] + giá_k)` với `k` thuộc `{1, 7, 30}` và `d - k` chặn dưới ở `0`.

### Checklist trước khi submit

- [ ] Ngày không đi: `dp[d] = dp[d - 1]`?
- [ ] Chặn chỉ số âm bằng `max(0, d - k)` (không dùng `dp[d - 7]` trần)?
- [ ] Đúng ba loại vé: `1`, `7`, `30` ngày với `costs[0], costs[1], costs[2]`?
- [ ] Độ dài vé đúng: vé 7 ngày dùng `dp[d - 7]` (không phải `d - 6`)?
- [ ] Không giả định vé dài đắt hơn vé ngắn (không tham lam)?
- [ ] Đã thử: một ngày, ngày `365`, đi cả năm, ngày cách xa, vé dài rẻ hơn vé ngắn?